# Notebook de Fine-Tuning de LLM Hugging Face para el Proyecto StyleAI
Este notebook contiene la guía interactiva paso a paso para reentrenar un modelo de lenguaje en Google Colab / Kaggle o GPU local utilizando **Hugging Face PEFT, QLoRA y SFTTrainer**.

## Paso 1: Instalación de Dependencias

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate torch

## Paso 2: Generar Dataset Instruccional

In [ ]:
# Ejecutar el script generador de dataset
!python dataset_generator.py

## Paso 3: Cargar Dataset y Verificar Estructura

In [ ]:
from datasets import load_dataset
dataset = load_dataset('json', data_files='data/styleai_dataset.jsonl', split='train')
print('Total de muestras:', len(dataset))
print('Muestra 0:', dataset[0])

## Paso 4: Cargar Modelo y Configurar QLoRA (Cuantización 4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Paso 5: Entrenamiento con SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./styleai-llm-adapter',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='steps',
    save_steps=50,
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field='messages',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

## Paso 6: Guardar Modelo Reentrenado y Probar Inferencia

In [ ]:
model.save_pretrained('./styleai-llm-adapter')
tokenizer.save_pretrained('./styleai-llm-adapter')
print('¡Adaptador de Fine-Tuning guardado en ./styleai-llm-adapter!')